# Agentick Benchmark Results — Debug Dashboard

Quick interactive exploration of **all** benchmark results: PPO, API LLMs, local LLMs, VLMs, baselines.

**Usage:** Run all cells. The loader auto-discovers all result directories under `results/`.

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## 1. Load & Merge All Result Directories

Auto-discovers result directories under all benchmark subdirs (`ppo_benchmarks`, `api_benchmarks`, `llm_benchmarks`, `vlm_benchmarks`, `full_benchmark`). Deduplicates and keeps latest per (config, task, difficulty).

In [ ]:
RESULTS_BASE = Path("results")

# All known benchmark subdirectories
RESULTS_SUBDIRS = [
    "ppo_benchmarks",
    "api_benchmarks",
    "llm_benchmarks",
    "vlm_benchmarks",
    "full_benchmark",
]

def load_run_dir(run_dir: Path, config_name: str, rows: list, training_curves: dict):
    """Load results from a single run directory into rows and training_curves.
    
    Prefers per_task/ directory (complete data from SLURM per-task jobs)
    over training_summary.json (which may only contain a single task).
    """
    loaded_keys = set()

    # Primary: load from per_task/ directory (has all tasks from SLURM jobs)
    per_task_dir = run_dir / "per_task"
    if per_task_dir.exists():
        for task_dir in per_task_dir.iterdir():
            if not task_dir.is_dir():
                continue
            for diff_dir in task_dir.iterdir():
                if not diff_dir.is_dir():
                    continue
                metrics_path = diff_dir / "metrics.json"
                if not metrics_path.exists():
                    continue
                with open(metrics_path) as f:
                    m = json.load(f)
                key = (m["task"], m["difficulty"])
                loaded_keys.add(key)
                rows.append({
                    "config": config_name,
                    "run_dir": str(run_dir),
                    "task": m["task"],
                    "difficulty": m["difficulty"],
                    "category": m.get("category", "unknown"),
                    "success_rate": m.get("success_rate", 0.0),
                    "mean_return": m.get("mean_return", 0.0),
                    "std_return": m.get("std_return", 0.0),
                    "mean_length": m.get("mean_length", 0.0),
                    "error": m.get("error"),
                })
                curve = m.get("training_curve", [])
                if curve:
                    training_curves[(config_name, m["task"], m["difficulty"])] = curve

    # Fallback: load from training_summary.json for any tasks not in per_task/
    ts_path = run_dir / "training_summary.json"
    if ts_path.exists():
        with open(ts_path) as f:
            ts = json.load(f)
        for key_name, result in ts.get("results", {}).items():
            key = (result["task"], result["difficulty"])
            if key in loaded_keys:
                continue  # already loaded from per_task/
            loaded_keys.add(key)
            rows.append({
                "config": config_name,
                "run_dir": str(run_dir),
                "task": result["task"],
                "difficulty": result["difficulty"],
                "category": result.get("category", "unknown"),
                "success_rate": result.get("success_rate", 0.0),
                "mean_return": result.get("mean_return", 0.0),
                "std_return": result.get("std_return", 0.0),
                "mean_length": result.get("mean_length", 0.0),
                "error": result.get("error"),
            })
            curve = result.get("training_curve", [])
            if curve:
                training_curves[(config_name, result["task"], result["difficulty"])] = curve

# --- Scan all result directories ---
rows = []
training_curves = {}

for subdir_name in RESULTS_SUBDIRS:
    subdir = RESULTS_BASE / subdir_name
    if not subdir.exists():
        continue
    for run_dir in sorted(subdir.iterdir()):
        if not run_dir.is_dir():
            continue
        # Derive config name by stripping timestamp suffix
        config_name = run_dir.name.rsplit("_2026", 1)[0]
        load_run_dir(run_dir, config_name, rows, training_curves)

# Also scan RESULTS_BASE directly for any flat result dirs
for run_dir in sorted(RESULTS_BASE.iterdir()):
    if not run_dir.is_dir() or run_dir.name in RESULTS_SUBDIRS:
        continue
    # Check if it looks like a result dir (has per_task/ or training_summary.json)
    if (run_dir / "per_task").exists() or (run_dir / "training_summary.json").exists():
        config_name = run_dir.name.rsplit("_2026", 1)[0]
        load_run_dir(run_dir, config_name, rows, training_curves)

df = pd.DataFrame(rows)

if df.empty:
    print("No results found! Check that RESULTS_BASE points to the right directory.")
else:
    # Deduplicate: keep the latest run per (config, task, difficulty)
    df = df.drop_duplicates(subset=["config", "task", "difficulty"], keep="last").reset_index(drop=True)
    
    print(f"Loaded {len(df)} task-difficulty results across {df['config'].nunique()} configs")
    print(f"Tasks: {df['task'].nunique()}, Difficulties: {sorted(df['difficulty'].unique())}")
    print(f"Errored runs: {df['error'].notna().sum()}")
    print(f"\nConfigs found:")
    for config in sorted(df["config"].unique()):
        n = df[df["config"] == config]["task"].nunique()
        print(f"  {config} ({n} tasks)")
    df.head()

## 2. Overview: Success Rate by Config

In [ ]:
for config in sorted(df["config"].unique()):
    sub = df[df["config"] == config]
    print(f"\n=== {config} ===")
    print(f"  Tasks covered: {sub['task'].nunique()} / 38")
    print(f"  Overall success rate: {sub['success_rate'].mean():.3f}")
    print(f"  Overall mean return:  {sub['mean_return'].mean():.3f}")
    for diff in ["easy", "medium", "hard", "expert"]:
        d = sub[sub["difficulty"] == diff]
        if len(d) > 0:
            print(f"  {diff:>8s}: success={d['success_rate'].mean():.3f}  return={d['mean_return'].mean():.3f}  (n={len(d)})")

## 3. Heatmap: Task × Difficulty (Success Rate)

In [ ]:
DIFF_ORDER = ["easy", "medium", "hard", "expert"]
CATEGORY_ORDER = [
    "navigation", "memory", "reasoning", "skill", "control",
    "combinatorial", "adversarial", "meta", "multi_agent", "compositional",
]

def plot_heatmap(df_config, config_name, metric="success_rate", cmap="RdYlGn"):
    # Sort tasks by category then name
    task_cat = df_config.drop_duplicates("task").set_index("task")["category"]
    cat_rank = {c: i for i, c in enumerate(CATEGORY_ORDER)}
    tasks_sorted = sorted(task_cat.index, key=lambda t: (cat_rank.get(task_cat[t], 99), t))
    
    pivot = df_config.pivot_table(index="task", columns="difficulty", values=metric, aggfunc="mean")
    pivot = pivot.reindex(index=tasks_sorted, columns=DIFF_ORDER)
    
    fig, ax = plt.subplots(figsize=(6, max(8, len(tasks_sorted) * 0.3)))
    im = ax.imshow(pivot.values, cmap=cmap, aspect="auto", vmin=0, vmax=1)
    
    ax.set_xticks(range(len(DIFF_ORDER)))
    ax.set_xticklabels(DIFF_ORDER)
    ax.set_yticks(range(len(tasks_sorted)))
    # Color ytick labels by category
    cat_colors = plt.cm.tab10(np.linspace(0, 1, 10))
    ylabels = ax.set_yticklabels(tasks_sorted, fontsize=8)
    for label, task in zip(ylabels, tasks_sorted):
        cat = task_cat.get(task, "unknown")
        cidx = cat_rank.get(cat, 0)
        label.set_color(cat_colors[cidx])
    
    # Annotate cells
    for i, task in enumerate(tasks_sorted):
        for j, diff in enumerate(DIFF_ORDER):
            val = pivot.iloc[i, j]
            if pd.notna(val):
                color = "white" if val < 0.5 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=7, color=color)
            else:
                ax.text(j, i, "—", ha="center", va="center", fontsize=7, color="gray")
    
    plt.colorbar(im, ax=ax, shrink=0.5, label=metric)
    ax.set_title(f"{config_name} — {metric}", fontsize=12)
    plt.tight_layout()
    plt.show()

for config in sorted(df["config"].unique()):
    plot_heatmap(df[df["config"] == config], config)

## 4. Per-Category Aggregated Success Rate

In [ ]:
cat_agg = df.groupby(["config", "category"]).agg(
    success_rate=("success_rate", "mean"),
    mean_return=("mean_return", "mean"),
    n_tasks=("task", "nunique"),
).reset_index()

configs = sorted(df["config"].unique())
n_configs = len(configs)

ncols = min(n_configs, 4)
nrows = (n_configs + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharey=True, squeeze=False)
for idx, config in enumerate(configs):
    ax = axes[idx // ncols][idx % ncols]
    sub = cat_agg[cat_agg["config"] == config].set_index("category")
    sub = sub.reindex(CATEGORY_ORDER)
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    ax.barh(range(len(CATEGORY_ORDER)), sub["success_rate"].fillna(0), color=colors)
    ax.set_yticks(range(len(CATEGORY_ORDER)))
    ax.set_yticklabels(CATEGORY_ORDER, fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Mean Success Rate", fontsize=8)
    ax.set_title(config, fontsize=9)
    ax.invert_yaxis()
    for i, cat in enumerate(CATEGORY_ORDER):
        val = sub.loc[cat, "success_rate"] if cat in sub.index and pd.notna(sub.loc[cat, "success_rate"]) else 0
        ax.text(val + 0.02, i, f"{val:.0%}", va="center", fontsize=7)

# Hide empty subplots
for idx in range(n_configs, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

plt.suptitle("Success Rate by Category", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Difficulty Scaling: Does Performance Drop as Expected?

In [ ]:
diff_agg = df.groupby(["config", "difficulty"]).agg(
    success_rate=("success_rate", "mean"),
    std=("success_rate", "std"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
palette = plt.cm.Set2(np.linspace(0, 1, max(len(configs), 2)))
for idx, config in enumerate(configs):
    sub = diff_agg[diff_agg["config"] == config].set_index("difficulty").reindex(DIFF_ORDER)
    x = range(len(DIFF_ORDER))
    ax.plot(x, sub["success_rate"], "o-", label=config, linewidth=2, markersize=5, color=palette[idx])
    ax.fill_between(x,
        (sub["success_rate"] - sub["std"]).clip(0),
        (sub["success_rate"] + sub["std"]).clip(0, 1),
        alpha=0.1, color=palette[idx])

ax.set_xticks(range(len(DIFF_ORDER)))
ax.set_xticklabels(DIFF_ORDER)
ax.set_ylabel("Mean Success Rate")
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=7, ncol=2)
ax.set_title("Difficulty Scaling (all tasks aggregated)")
plt.tight_layout()
plt.show()

## 6. Per-Task Success Rate Bars (Sorted)

In [ ]:
for config in configs:
    sub = df[df["config"] == config]
    task_avg = sub.groupby("task").agg(
        success_rate=("success_rate", "mean"),
        category=("category", "first"),
    ).sort_values("success_rate", ascending=True)
    
    cat_colors_map = {c: plt.cm.tab10(i / 10) for i, c in enumerate(CATEGORY_ORDER)}
    colors = [cat_colors_map.get(task_avg.loc[t, "category"], "gray") for t in task_avg.index]
    
    fig, ax = plt.subplots(figsize=(8, max(6, len(task_avg) * 0.25)))
    bars = ax.barh(range(len(task_avg)), task_avg["success_rate"], color=colors)
    ax.set_yticks(range(len(task_avg)))
    ax.set_yticklabels(task_avg.index, fontsize=7)
    ax.set_xlim(0, 1)
    ax.set_xlabel("Mean Success Rate (across difficulties)")
    ax.set_title(f"{config} — Per-Task Success Rate", fontsize=11)
    
    # Annotate values
    for i, (task, row) in enumerate(task_avg.iterrows()):
        ax.text(row["success_rate"] + 0.01, i, f"{row['success_rate']:.0%}", va="center", fontsize=7)
    
    # Legend for categories
    from matplotlib.patches import Patch
    present_cats = task_avg["category"].unique()
    legend_patches = [Patch(color=cat_colors_map.get(c, "gray"), label=c) for c in CATEGORY_ORDER if c in present_cats]
    ax.legend(handles=legend_patches, loc="lower right", fontsize=7, ncol=2)
    
    plt.tight_layout()
    plt.show()

## 7. Learning Curves (Per Task, Grid of Subplots)

In [ ]:
for config in configs:
    # Get all tasks that have training curves for this config
    config_curves = {(t, d): v for (c, t, d), v in training_curves.items() if c == config}
    tasks_with_curves = sorted(set(t for t, d in config_curves.keys()))
    
    if not tasks_with_curves:
        print(f"No training curves for {config}")
        continue
    
    n = len(tasks_with_curves)
    ncols = 6
    nrows = (n + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.2), squeeze=False)
    diff_colors = {"easy": "#2ca02c", "medium": "#ff7f0e", "hard": "#d62728", "expert": "#7f7f7f"}
    
    for idx, task in enumerate(tasks_with_curves):
        ax = axes[idx // ncols][idx % ncols]
        for diff in DIFF_ORDER:
            curve = config_curves.get((task, diff), [])
            if not curve:
                continue
            steps = [p["timestep"] for p in curve]
            rewards = [p["mean_reward"] for p in curve]
            stds = [p.get("std_reward", 0) for p in curve]
            ax.plot(np.array(steps) / 1000, rewards, label=diff, color=diff_colors[diff], linewidth=1)
            ax.fill_between(
                np.array(steps) / 1000,
                np.array(rewards) - np.array(stds),
                np.array(rewards) + np.array(stds),
                alpha=0.1, color=diff_colors[diff],
            )
        ax.set_title(task.replace("-v0", ""), fontsize=7)
        ax.tick_params(labelsize=6)
        ax.set_xlabel("steps (k)", fontsize=6)
    
    # Hide empty subplots
    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)
    
    # Single legend
    handles, labels = axes[0][0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right", fontsize=8)
    fig.suptitle(f"{config} — Learning Curves", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 8. Failure Analysis: Errored & Zero-Success Tasks

In [ ]:
for config in configs:
    sub = df[df["config"] == config]
    
    # Errored tasks
    errored = sub[sub["error"].notna()]
    if len(errored) > 0:
        print(f"\n=== {config}: ERRORED RUNS ({len(errored)}) ===")
        for _, row in errored.iterrows():
            print(f"  {row['task']:30s} {row['difficulty']:8s}  {row['error'][:80]}")
    
    # Zero success on easy (should be learnable!)
    easy = sub[(sub["difficulty"] == "easy") & (sub["success_rate"] == 0.0) & (sub["error"].isna())]
    if len(easy) > 0:
        print(f"\n=== {config}: ZERO SUCCESS ON EASY ({len(easy)} tasks) ===")
        for _, row in easy.iterrows():
            print(f"  {row['task']:30s}  return={row['mean_return']:.3f}")
    
    # Perfect success on all difficulties (sanity check)
    perfect = sub.groupby("task").filter(lambda g: (g["success_rate"] == 1.0).all())
    if len(perfect) > 0:
        tasks = perfect["task"].unique()
        print(f"\n=== {config}: PERFECT ON ALL DIFFICULTIES ({len(tasks)} tasks) ===")
        for t in sorted(tasks):
            print(f"  {t}")

## 9. Dense vs Sparse Comparison

In [ ]:
dense_configs = [c for c in configs if "dense" in c]
sparse_configs = [c for c in configs if "sparse" in c]

if dense_configs and sparse_configs:
    dense_name = dense_configs[0]
    sparse_name = sparse_configs[0]
    
    dense_df = df[df["config"] == dense_name].set_index(["task", "difficulty"])["success_rate"]
    sparse_df = df[df["config"] == sparse_name].set_index(["task", "difficulty"])["success_rate"]
    
    # Align on common tasks
    common = dense_df.index.intersection(sparse_df.index)
    if len(common) > 0:
        comparison = pd.DataFrame({
            "dense": dense_df.reindex(common),
            "sparse": sparse_df.reindex(common),
        }).dropna()
        comparison["diff"] = comparison["dense"] - comparison["sparse"]
        
        # Per-task average (numeric_only to avoid string column error)
        task_comp = comparison.reset_index().groupby("task")[["dense", "sparse", "diff"]].mean()
        task_comp = task_comp.sort_values("diff")
        
        fig, ax = plt.subplots(figsize=(8, max(6, len(task_comp) * 0.25)))
        y = range(len(task_comp))
        ax.barh(y, task_comp["dense"], height=0.4, label="Dense", color="#2ca02c", align="edge")
        ax.barh([yi - 0.4 for yi in y], task_comp["sparse"], height=0.4, label="Sparse", color="#d62728", align="edge")
        ax.set_yticks(y)
        ax.set_yticklabels(task_comp.index, fontsize=7)
        ax.set_xlim(0, 1)
        ax.set_xlabel("Mean Success Rate")
        ax.set_title("Dense vs Sparse Reward")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("No overlapping tasks between dense and sparse configs.")
else:
    print("Need both dense and sparse configs for comparison.")

In [ ]:
if len(configs) >= 2:
    cat_scores = df.groupby(["config", "category"])["success_rate"].mean().unstack(level=1)
    cat_scores = cat_scores.reindex(columns=CATEGORY_ORDER).fillna(0)
    
    angles = np.linspace(0, 2 * np.pi, len(CATEGORY_ORDER), endpoint=False).tolist()
    angles += angles[:1]  # close the polygon
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    palette = plt.cm.Set2(np.linspace(0, 1, len(configs)))
    
    for idx, config in enumerate(configs):
        values = cat_scores.loc[config].tolist()
        values += values[:1]
        ax.plot(angles, values, "o-", linewidth=1.5, label=config, color=palette[idx], markersize=4)
        ax.fill(angles, values, alpha=0.08, color=palette[idx])
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(CATEGORY_ORDER, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["25%", "50%", "75%", "100%"], fontsize=7)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=7)
    ax.set_title("Capability Radar: All Agents", fontsize=12, y=1.08)
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 configs for radar comparison.")

## 9c. Capability Radar: All Agents Overlaid

Each spoke is a task category (10 categories). Each agent is a polygon.

In [ ]:
if len(configs) >= 2:
    # Build task × agent matrix (mean success rate across difficulties)
    agent_task = df.groupby(["config", "task"])["success_rate"].mean().unstack(level=0)
    
    # Sort tasks by category
    task_cat = df.drop_duplicates("task").set_index("task")["category"]
    cat_rank = {c: i for i, c in enumerate(CATEGORY_ORDER)}
    tasks_sorted = sorted(agent_task.index, key=lambda t: (cat_rank.get(task_cat.get(t, "unknown"), 99), t))
    agent_task = agent_task.reindex(index=tasks_sorted)
    
    fig, ax = plt.subplots(figsize=(max(8, len(configs) * 1.2), max(8, len(tasks_sorted) * 0.3)))
    im = ax.imshow(agent_task.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
    
    ax.set_xticks(range(len(agent_task.columns)))
    ax.set_xticklabels(agent_task.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(tasks_sorted)))
    
    # Color ytick labels by category
    cat_colors_arr = plt.cm.tab10(np.linspace(0, 1, 10))
    ylabels = ax.set_yticklabels(tasks_sorted, fontsize=7)
    for label, task in zip(ylabels, tasks_sorted):
        cat = task_cat.get(task, "unknown")
        cidx = cat_rank.get(cat, 0)
        label.set_color(cat_colors_arr[cidx])
    
    # Annotate cells
    for i, task in enumerate(tasks_sorted):
        for j, config in enumerate(agent_task.columns):
            val = agent_task.iloc[i, j]
            if pd.notna(val):
                color = "white" if val < 0.5 else "black"
                ax.text(j, i, f"{val:.0%}", ha="center", va="center", fontsize=6, color=color)
    
    plt.colorbar(im, ax=ax, shrink=0.4, label="Success Rate")
    ax.set_title("Cross-Agent Comparison: Task × Agent (mean across difficulties)", fontsize=11)
    plt.tight_layout()
    plt.show()
    
    # Overall ranking
    print("\nOverall mean success rate:")
    overall = df.groupby("config")["success_rate"].mean().sort_values(ascending=False)
    for config, sr in overall.items():
        print(f"  {config:<45s} {sr:.1%}")
else:
    print("Need at least 2 configs for cross-agent comparison.")

## 9b. Cross-Agent Comparison Heatmap

All agents side-by-side on a single heatmap (tasks × agents), averaged across difficulties.

## 10. Missing Tasks (Not Yet Run or Lost)

In [ ]:
TASK = "GoToGoal-v0"  # <-- change this
CONFIG = configs[0] if configs else ""  # <-- change this

print(f"Available configs: {configs}")
print(f"Available tasks: {sorted(df['task'].unique())[:10]}... ({df['task'].nunique()} total)")
print()

task_df = df[(df["task"] == TASK) & (df["config"] == CONFIG)]
if task_df.empty:
    print(f"No data for {TASK} in {CONFIG}")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Left: success rate by difficulty
    ax = axes[0]
    task_df_sorted = task_df.set_index("difficulty").reindex(DIFF_ORDER)
    colors = ["#2ca02c", "#ff7f0e", "#d62728", "#7f7f7f"]
    ax.bar(DIFF_ORDER, task_df_sorted["success_rate"], color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Success Rate")
    ax.set_title(f"{TASK} — Success Rate")
    for i, diff in enumerate(DIFF_ORDER):
        val = task_df_sorted.loc[diff, "success_rate"]
        ax.text(i, val + 0.02, f"{val:.0%}", ha="center", fontsize=10)
    
    # Right: learning curves (if available)
    ax = axes[1]
    diff_colors_map = dict(zip(DIFF_ORDER, colors))
    has_curves = False
    for diff in DIFF_ORDER:
        curve = training_curves.get((CONFIG, TASK, diff), [])
        if curve:
            has_curves = True
            steps = [p["timestep"] / 1000 for p in curve]
            rewards = [p["mean_reward"] for p in curve]
            stds = [p.get("std_reward", 0) for p in curve]
            ax.plot(steps, rewards, label=diff, color=diff_colors_map[diff], linewidth=1.5)
            ax.fill_between(steps,
                np.array(rewards) - np.array(stds),
                np.array(rewards) + np.array(stds),
                alpha=0.15, color=diff_colors_map[diff])
    if has_curves:
        ax.set_xlabel("Steps (k)")
        ax.set_ylabel("Mean Reward")
        ax.set_title(f"{TASK} — Learning Curves")
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No training curves\n(non-RL agent)", ha="center", va="center",
                transform=ax.transAxes, fontsize=12, color="gray")
        ax.set_title(f"{TASK} — Learning Curves (N/A)")
    
    plt.tight_layout()
    plt.show()
    
    # Print raw metrics
    print(f"\n{TASK} ({CONFIG}):")
    for _, row in task_df.iterrows():
        print(f"  {row['difficulty']:8s}: success={row['success_rate']:.0%}  "
              f"return={row['mean_return']:.3f} +/- {row['std_return']:.3f}  "
              f"length={row['mean_length']:.1f}")

## 11. Quick Summary Table

In [ ]:
summary_rows = []
for config in configs:
    sub = df[df["config"] == config]
    for diff in DIFF_ORDER:
        d = sub[sub["difficulty"] == diff]
        summary_rows.append({
            "config": config,
            "difficulty": diff,
            "n_tasks": d["task"].nunique(),
            "mean_success": d["success_rate"].mean(),
            "n_solved": (d["success_rate"] > 0).sum(),
            "n_perfect": (d["success_rate"] == 1.0).sum(),
            "mean_return": d["mean_return"].mean(),
        })

summary_df = pd.DataFrame(summary_rows)
summary_df["mean_success"] = summary_df["mean_success"].map("{:.1%}".format)
summary_df["mean_return"] = summary_df["mean_return"].map("{:.3f}".format)
display(summary_df)

## 12. Drill Down: Single Task Inspector

Change `TASK` and `CONFIG` below to zoom into a specific task.

In [ ]:
TASK = "GoToGoal-v0"  # <-- change this
CONFIG = configs[0] if configs else ""  # <-- change this

task_df = df[(df["task"] == TASK) & (df["config"] == CONFIG)]
if task_df.empty:
    print(f"No data for {TASK} in {CONFIG}")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Left: success rate by difficulty
    ax = axes[0]
    task_df_sorted = task_df.set_index("difficulty").reindex(DIFF_ORDER)
    colors = ["#2ca02c", "#ff7f0e", "#d62728", "#7f7f7f"]
    ax.bar(DIFF_ORDER, task_df_sorted["success_rate"], color=colors)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("Success Rate")
    ax.set_title(f"{TASK} — Success Rate")
    for i, diff in enumerate(DIFF_ORDER):
        val = task_df_sorted.loc[diff, "success_rate"]
        ax.text(i, val + 0.02, f"{val:.0%}", ha="center", fontsize=10)
    
    # Right: learning curves
    ax = axes[1]
    diff_colors_map = dict(zip(DIFF_ORDER, colors))
    for diff in DIFF_ORDER:
        curve = training_curves.get((CONFIG, TASK, diff), [])
        if curve:
            steps = [p["timestep"] / 1000 for p in curve]
            rewards = [p["mean_reward"] for p in curve]
            stds = [p.get("std_reward", 0) for p in curve]
            ax.plot(steps, rewards, label=diff, color=diff_colors_map[diff], linewidth=1.5)
            ax.fill_between(steps,
                np.array(rewards) - np.array(stds),
                np.array(rewards) + np.array(stds),
                alpha=0.15, color=diff_colors_map[diff])
    ax.set_xlabel("Steps (k)")
    ax.set_ylabel("Mean Reward")
    ax.set_title(f"{TASK} — Learning Curves")
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
    # Print raw metrics
    print(f"\n{TASK} ({CONFIG}):")
    for _, row in task_df.iterrows():
        print(f"  {row['difficulty']:8s}: success={row['success_rate']:.0%}  "
              f"return={row['mean_return']:.3f} +/- {row['std_return']:.3f}  "
              f"length={row['mean_length']:.1f}")